# report12 — 다중 수신기 디텍션 + 9-모드 벤치마크

**핵심.** 앞 리포트의 무대·표적(SBR σ)·신호(Sionna PHY)·검출기를 하나로 합쳐, **어떤 통신 신호로 · 수신기 몇 개로 드론이 실제로 잡히는지**를 몬테카를로로 못박는다 — 9모드(3표준×3점유) × 감시 배열 1→4, K=2,000회. 이 프로젝트의 **결과편**이다(탐지까지 — 추적은 future work).

| 이 리포트의 척추 |  |
|---|---|
| **① Sionna 의 공백** | 탐지 한 번은 네 재료를 요구한다: **① 조명 파형 · ② 채널(τ·f_d) · ③ 표적 밝기 σ · ④ 검출 체인**(직접파 제거 ECA → 거리-도플러 → CFAR → 빔포밍). Sionna 는 **①②만** 담보하고(PHY·RT), **③ σ 는 산란적분 부재로 못 내며**(report06), **④ 패시브 레이더 검출 파이프라인은 아예 없다**(Sionna 에 레이더 DSP 부재). 셋을 밖에서 채워 결합해야 한다. |
| **② 선행 연구의 방식** | 패시브 바이스태틱 드론탐지 선행은 표적 산란을 **외부에서 구해 채널에 주입**($h=h_{bg}+h_{target}$; NIST 5GNRad · 3GPP Rel-19 오픈구현(arXiv:2606.07328) · LAMBDA=Sionna+CADFEKO(arXiv:2607.03826))하고, 표준 검출 체인(ECA/CAF/CFAR)으로 잡는다(Wypich Sensors 2026 · Demissie IET RSN 2025). 단 조사한 선행은 대개 단일 조명원·최대 2채널이라 **표준 간·다중 Rx 공정 비교가 비어 있다**(§6). |
| **③ 쓴 라이브러리·결합** | 검출 체인은 **pyAPRiL**(GPLv3, 실측 검증된 패시브 레이더 ECA/CAF/CFAR)로 정확성을 검증하고, 대량 Pd 곡선은 그 검증된 체인을 **GPU(torch)로 K=2,000회 반복**한다(`detection_gpu.py`). 표적 에코의 지연 펄스정형은 **Sionna PHY `cir_to_time_channel`** 커널을 그대로, 표적 밝기 σ 는 **SBR+PO** — 새 파형·산란 엔진을 만들지 않고 검증된 조각만 결합한다(중복계산 없음). |
| **④ 검증** | **GPU 배치 몬테카를로 K=2,000회**(각 SNR·N·모드)로 Pd. pyAPRiL 로 NR/WiFi/LTE 3모드 모두 **CAF 봉우리 거리빈이 정답과 일치**함을 확인(`verify_pyapril.py`; ⚠CFAR 표적셀 발화는 아님); 지연 연산자는 해석식과 상관 1.000(지연만), 배열이득 √N 은 잡음전력 보존 1.000만 독립 실측. 절대 σ 는 실측 문헌 드론 RCS 로 앵커(report08). |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| Pd·SNR50·경험적 Pfa (9모드×N=1..4×SNR) | outputs/detection_rx_sweep.json | 측정 (몬테카를로 K회, GPU 배치) |
| 표적 밝기 σ | src/rcs_po.py → SBR (Mitsuba 광선 + PO) | 측정 (mavic4pro, σ=-28.0 dBsm) |
| 표적 에코 (지연·도플러) | src/sionna_chain.py → cir_to_time_channel (Sionna PHY) | 측정 (해석식과 상관 1.000) |
| 9모드 파형(3표준×3점유) | src/waveforms.py (occupancy G1/G2/G3) · sionna.phy.nr | 🔴 우리 구현 + 🟢 Sionna 뉴머롤로지 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sionna-phy` | Sionna PHY (`ofdm`/`nr`/`channel`) — OFDM 변복조 · 3GPP 뉴머롤로지 · RT 경로를 신호에 적용 | 🟢 **Sionna 내부** (PyTorch 백엔드, GPU) |
| `sbr` | SBR (`src/rcs_sbr.py`) — **Mitsuba 광선 + PO 표면적분**으로 RCS. 가림(occlusion) 포함 | 🟡 **우리가 짰다** — 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진 그대로** (GPU). Sionna 에 RCS 솔버가 없기 때문 |
| `radar-dsp` | 레이더 신호처리 (`src/passive_process.py`) — ECA(직접파 제거) · 거리-도플러 · CA-CFAR | 🔴 **별도** (numpy, CPU). **Sionna 에 레이더 DSP 는 없다** |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 얹은 **PO(물리광학 표면적분)** 가 냅니다 — Sionna 기본 solver 엔 이 산란적분이 없어 경로 이득만 줄 뿐 RCS 를 못 내기 때문입니다. 광선을 쏴 조명면·가림을 찾는 **SBR** 은 Sionna 의 **Mitsuba 3 엔진을 그대로** 쓰고, 그 위에 **PO 적분만 우리가** 얹습니다(SBR+PO).
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `torch` | 2.12.1 | Sionna PHY 백엔드 — ⚠ Sionna 2.0 은 TensorFlow 가 아니라 **PyTorch** |
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `mitsuba` | 3.8.0 | Sionna RT 의 렌더러·광선추적 백엔드 (OptiX, GPU). SBR 도 이걸 쓴다 |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: GPU 배치 몬테카를로: 9모드×4×15×K 트라이얼 — 카드 1장에서 수십 분(배치 크게 → GPU 메모리 대량 사용).

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2
SIONNA2_GPU=3 ~/.venvs/py312/bin/python src/detection_gpu.py    # GPU 커널 자기검증
SIONNA2_GPU=3 SIONNA2_DET_BATCH=48 ~/.venvs/py312/bin/python src/experiment_detection.py  # 9모드 스윕
~/.venvs/py312/bin/python src/anim_plots.py --which all         # 애니(GIF)
~/.venvs/py312/bin/python src/make_notebook12.py                # report12.ipynb
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/detection_rx_sweep.json` | 9모드 Pd/Pfa/SNR50 전체 스윕 |
| `outputs/figures/report12_9mode.png` | 9-모드 벤치마크 (헤드라인) |
| `outputs/figures/report12_pd_curves.png` | Rx 1→4 Pd 곡선 (풀 모드) |
| `outputs/renders/anim/rd_rxbuildup_nr.gif` | Rx 증설로 표적이 떠오르는 RD 맵 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **'점유율'을 시간가변으로 현실성 있게 모델하진 않았다**(사용자와 합의). 9모드는 **고정 점유 레벨**이며, 각 모드는 그 모드의 송신 파형을 기준으로 상관한다(full-waveform capture 상한). WiFi 센싱에서 트래픽을 인위로 늘려 재는 관행과 같은 전제다.
- **상시 vs 제어는 '기준신호 종류'의 차이지 순수 점유율이 아니다.** LTE 는 L1=CRS→L2·L3=PRS, 5G 는 G1=SSB→G2·G3=NR-PRS 로 **기준신호 자체가 바뀐다**. WiFi 는 셋 다 프리앰블이라 분해능이 거의 안 변한다(데이터 점유만 다름).
- **코히어런트 결합은 소자 간 위상을 안다고 가정**한다. 몬테카를로는 √N 을 해석적으로 주입하고 잡음 보존만 실측하므로, 보고된 +6 dB(N=4)는 **이상적 상한**이다 — 실측(X410)은 조향 불일치·보정오차로 그 이하.
- **Pfa 공정성 미완(적대적 검증이 짚음).** 파형(std)별 명목 Pfa 만 교정해서 **모드 간 경험적 Pfa 가 완전히 같지 않다**(특히 5G 상시 SSB 는 대역이 좁아 미교정). 큰 모드 차이(대역폭 물리)는 견고하나, ~1 dB 안쪽 차이는 이 한계를 감안해 읽어야 한다. 완전 공정 비교는 경험적 Pfa 축 ROC 가 필요(향후 과제).
- **5G 이중고 중 '저반복' 축은 이 실험에 안 들어갔다(정직).** CPI 는 표준별 고정 PRF(fs/Lf)로 타일링하는데, 5G 상시신호(SSB)의 **실제 반복률은 그보다 훨씬 낮다**(SSB 는 20 ms 주기라 무모호 속도 ~1 m/s). 즉 '5G 이중고 = 좁은 대역 + 낮은 반복' 중 **대역(거리분해능) 쪽만** 실험에 반영됐고 **반복(빠른 표적 접힘) 쪽은 낙관적**이다. 표적이 느려(도플러 64 Hz) **이 실험에 쓴 높은 PRF 에서는** 안 접히므로 (실 SSB 반복률 20 ms 에서는 접힌다 — 그건 G1 을 더 불리하게 하므로 헤드라인 결론은 오히려 강화된다) **현재 결론(G1 이 대역 때문에 최악)은 유효**하지만, 빠른 표적에선 5G 가 더 불리하다(→ report11 §2 모호함수).
- **이 리포트는 탐지(있다/없다)까지다.** 위치·궤적을 잇는 **추적은 다음 일**이며, 감시 배열의 각도(AoA)로 관측가능성을 확보해야 한다(report11).

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| report04 (앞) | 9모드 파형·상시 기준신호·5G 이중고 — 이 벤치마크의 신호들 |
| report07~08·10~11 (앞) | SBR σ · Pfa 교정 · 저속·단일 Rx 3D 불가 — 딛는 토대 |
| (끝) | 결과편. 추적은 future work. |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **패시브(수동) 바이스태틱 레이더** | 자기 송신기 없이 **이미 켜진 통신 신호**(WiFi·LTE·5G)를 조명으로 빌려, 떨어진 수신기로 표적 메아리를 듣는 레이더. |
| **ISAC (통합 센싱·통신)** | 송신기를 **통신+센싱 겸용으로 설계·제어**하는 것. 파형을 센싱에 맞게 빚을 수 있다. 패시브(제어 못 함)와 대비된다. 이 프로젝트의 X410 실험은 송신을 **제어**하므로 ISAC 에 가깝다. |
| **상시(always-on) vs 세션/제어 기준신호** | 상시 = 아무 셀이나 늘 내보내는 신호(WiFi 프리앰블·LTE CRS·5G SSB) — 협조 없이 패시브가 얻는 것. 세션/제어 = PRS(측위세션 필요) 또는 송신 전체 캡처(협조/제어) — 더 넓은 대역을 준다. |
| **검출확률 Pd / 오경보율 Pfa** | Pd=표적 있을 때 잡을 확률, Pfa=없는데 잘못 외칠 확률. **같은 Pfa 에서** Pd 를 비교해야 공정. |
| **RD 맵 (거리-도플러 지도)** | 수신 신호를 가로=거리, 세로=속도(도플러)로 펼친 2차원 지도. 표적은 자기 칸에 밝은 점. |
| **코히어런트 빔포밍** | 여러 수신 소자를 표적 방향으로 위상 맞춰 더하기. 표적 √N 배·잡음 그대로 → SNR N배(+10log10 N dB). |
| **거리분해능 (바이스태틱 vs 모노스태틱)** | 두 물체를 거리로 가르는 능력. 대역폭이 넓을수록 작아진다(좋다). ⚠ **바이스태틱**(우리 시스템)은 거리가 왕복이 아니므로 $\Delta R_b = c/B$, **모노스태틱**은 왕복이라 $\Delta R = c/2B$(절반). 두 값을 구분해 관리한다(report11 §2 도 바이스태틱 c/B). |
| **Sionna PHY / SBR** | Sionna PHY=파형·채널(여기서 표적 에코 생성). SBR=Mitsuba 광선+우리 PO 로 표적 밝기 σ. [[report06]][[report07]] |

</details>

---


# §1. Sionna 의 공백 — 검출 파이프라인이 없다

탐지 한 번은 네 재료를 요구한다: **① 조명 파형**, **② 표적까지의 채널**(지연 τ·도플러 f_d), **③ 표적 밝기 σ**, **④ 검출 체인**(직접파 제거 ECA → 거리-도플러 → CFAR → 빔포밍). Sionna 는 이 중 **①②만** 담보한다 — PHY 파형·`cir_to_time_channel` 지연커널과 RT 챔버 전파(report05 NMSE −135 dB). **③ σ 는 산란적분이 없어 못 내고**(report06), **④ 패시브 레이더 검출 파이프라인은 아예 없다**(Sionna 에 레이더 DSP 부재). 이 리포트는 ③을 SBR+PO 로, ④를 검증된 표준 체인으로 채워 넷을 결합한 뒤 K=2,000회 몬테카를로로 탐지를 측정한다.

그래서 "Sionna 로 디텍션을 했다"는 정확하지 않다. 정확히는 **Sionna 가 담보하는 전파 환경·파형 위에 표준 레이더 검출 체인을 올렸다**이다. 검출 몬테카를로의 각 층이 어디서 오는지 펼치면:

| 층 | 출처 | Sionna 관여 |
|---|---|---|
| 조명 파형(WiFi/LTE/5G OFDM) | Sionna PHY 뉴머롤로지(`sionna.phy.nr`) | ✅ 검증됨 — NMSE −135 dB(report05) |
| 표적 기하(R₁·R₂·L → τ·f_d) | 해석 기하 | RT 경로 기하와 일치 검증(report01·09) |
| 표적 밝기 σ | SBR+PO(선행 BVH SBR+PO 와 같은 방법) | **Sionna 불가**(산란적분 없음 — report06) |
| 표적 에코 진폭 | 바이스태틱 레이더 방정식 | — |
| 지연 펄스정형 | **Sionna PHY** `cir_to_time_channel` | ✅ 진짜 Sionna |
| 도플러 | 위상램프(시변 채널과 동등) | — |
| 직접파(DPI)·클러터 | 반무향 챔버 → 백색잡음+DPI(정적 클러터는 ECA 가 차단, report09) | 챔버 전파·바닥유령은 RT 로 별도 검증 |
| ECA → 거리-도플러 → CFAR → 빔포밍 | **표준 패시브 레이더 체인**(오픈소스 pyAPRiL 로 검증) | 시뮬레이터가 제공하지 않는 층 |

이 구조(σ 를 외부에서 주입 + 표적/배경 채널 분리 + 표준 검출 체인)는 **레이더 시뮬레이션의 표준**이다. 다른 부류의 도구가 각각 어디까지 하는지 보면 분명하다:

| 도구 부류 | ① 표적 밝기 σ | ② 채널(τ·f_d·감쇠) | ③ 검출 체인 |
|---|---|---|---|
| 드론 시뮬(Gazebo·AirSim·Isaac) | ✗ | ✗ | ✗ '광선 맞으면 탐지'로 **가정**(전파 물리 없음) |
| 레이더 시뮬 표준(MATLAB Radar Toolbox·NIST 5GNRad) | 상수/외부 EM 입력(`MeanRCS=`) | 해석식(자유공간) 또는 점산란체 | 블록 조립(CFAR 등) |
| EM 솔버(HFSS SBR+·Feko) | ✅ 이것만 | ✗ | ✗ |
| **이 스택** | SBR+PO(평판/구 이론 + 실측 문헌 앵커로 검증, report07·08) | 해석식 + **Sionna RT 가 챔버 물리를 담보** | 표준 ECA/CAF/CFAR(pyAPRiL 검증) |

즉 'σ 주입 + 표적/배경 채널 분리 + 표준 신호처리'는 편법이 아니라 **MATLAB·NIST 5GNRad 와 동일한 주류 구조**이고, 차이는 ②에서 자유공간 가정 대신 **Sionna RT 검증**을 얹은 것이다.

**Sionna 를 쓴 의미**는 검출 루프 안이 아니라 그 밖 세 곳에 있다. ① **환경 담보** — 이 시뮬의 채널이 '직접파+에코+잡음'뿐이어도 되는 근거는 가정이 아니라 RT 로 챔버를 추적한 결론이다(벽·천장 흡수·잔향 무시 가능·반사 경로는 바닥뿐, report01). ② **구멍 발견** — 해석 모델로는 몰랐을 표적경유 바닥 유령을 RT 가 찾아냈고(report09), 그것이 '전대역 5G 100% 오검출'이라는 결론이 됐다. ③ **실측 준비** — 실외 X410 은 멀티패스가 풍부해 해석 탭으로 안 되고, 그때 RT 가 채널 탭을 공급한다. 정확한 문장은 "Sionna 가 검출을 해줬다"가 아니라 **"Sionna 가 검출 시뮬의 전제를 검증하고 구멍(유령)을 찾았다"**이다.

> ⚠️ **남은 정직한 갭:** 이 몬테카를로의 채널은 '해석 탭(에코+직접파)+백색잡음'이다 — report09 의 바닥 유령 탭·잔향 꼬리는 이 MC 채널에 넣지 않았다(유령은 기하·위협 정량화까지). 실측 X410 대조에서 이 미모델 성분이 첫 용의자다.

# §2. 무대와 신호 — 증거를 만드는 재료

송신기(TX, 조명원)와 수신기가 **떨어져** 있다. 수신기는 두 몫: **기준 채널**(직접파를 받아 '무슨 신호를 쐈는지' 앎)과 **감시 배열**(표적 쪽을 보는 여러 소자, λ/2 균일선형배열). 감시 소자 수 **N 을 1→4** 로 늘리는 것이 실험 변수다.

표적 = **mavic4pro**(방위 az=-24°), SBR(Mitsuba) 밝기 **σ=-28.0 dBsm** (이 **특정 자세**의 값 — 방위평균은 −18.4 dBsm(3.5 GHz)·밴드평균 −17.1 dBsm, report08; RCS 는 자세에 ~14 dB 출렁인다), 속도 3 m/s → 도플러 ≈64 Hz, 바이스태틱 거리 R_b≈22.3 m.

![감시배열 Rx 1→4 증설](outputs/renders/anim/rx_array.gif)

<sub>감시 배열을 1→2→3→4 소자로 늘리는 모습(시각화용으로 간격 과장). 실제 λ/2=4.3 cm.</sub>

**표적 에코 만들기.** 표적 메아리 = 송신 파형이 표적에 맞고(지연 τ) 되쏘며(도플러 f_d), 세기는 SBR σ 로 정해진다. 파형과 지연 채널은 **Sionna PHY** 가 담당한다(검증됨, report05 NMSE −135 dB).

지연 τ 의 **분수지연 sinc 커널을 Sionna `cir_to_time_channel` 이 생성**하고, 표적 에코는 이 커널로 만든다. 도플러·진폭은 위상램프로 얹는데, 고정지연·협대역에서 시변 채널과 동등하다. 해석식과의 상관 **1.000** 로 **지연 연산자**를 교차검증한다(도플러·진폭은 양쪽이 같은 식이라 자명).

> **ISAC 접점:** 통신 쪽(C)은 Sionna PHY 의 파형·지연 펄스정형이 담보하고, 센싱 쪽(S)의 표적 밝기 σ 는 Sionna 가 못 내므로(산란적분 부재, report06) SBR+PO 로 얹는다 — LAMBDA·3GPP 가 쓰는 'Sionna 전파 + 외부 RCS 주입'과 같은 아키텍처. ([[report06]][[report07]])

# §3. 9-모드 벤치마크 — 어떤 신호가 드론을 잘 비추나

3표준 × 3점유(상시→제어/풀) = 9모드. 필요한 SNR(단일 Rx, 낮을수록 좋음):

| 모드 | 표준 | 기준신호 | 대역 $B_{ref}$ | $\Delta R_b$ (바이스태틱, c/B) | $\Delta R$ (모노 등가, c/2B) | 종류 | 필요 SNR |
|---|---|---|---|---|---|---|---|
| W1 | WiFi | VHT-LTF | 76.56 MHz | 3.9 m | 2.0 m | 상시 | 11.8 dB |
| W2 | WiFi | VHT-LTF | 76.56 MHz | 3.9 m | 2.0 m | 세션/제어 | 11.5 dB |
| W3 | WiFi | VHT-LTF | 76.56 MHz | 3.9 m | 2.0 m | 세션/제어 | 11.6 dB |
| L1 | LTE | CRS | 17.98 MHz | 16.7 m | 8.3 m | 상시 | 13.7 dB |
| L2 | LTE | PRS | 18.02 MHz | 16.6 m | 8.3 m | 세션/제어 | 13.6 dB |
| L3 | LTE | PRS | 18.02 MHz | 16.6 m | 8.3 m | 세션/제어 | 13.7 dB |
| G1 | 5G | SSB | 7.20 MHz | 41.6 m | 20.8 m | 상시 | 15.1 dB |
| G2 | 5G | NR-PRS | 98.28 MHz | 3.1 m | 1.5 m | 세션/제어 | 11.1 dB |
| G3 | 5G | NR-PRS | 98.28 MHz | 3.1 m | 1.5 m | 세션/제어 | 11.2 dB |

![9-mode benchmark SNR50](outputs/figures/report12_9mode.png)

**읽는 법.** 막대가 낮을수록 더 약한 드론도 잡는다(좋다). 빗금친 막대가 **상시(협조 없이 얻는 것)**.

> **거리분해능 규약(정합성):** 이 표의 $\Delta R_b = c/B_{ref}$ 는 **바이스태틱** 거리분해능이다(바이스태틱 거리는 왕복이 아니므로 $c/2B$ 가 아니라 $c/B$ — report11 §2 및 문헌 25_UAV Intrusion 과 동일 규약). 모노스태틱 등가값은 이 절반이다.

- **WiFi 는 상시라도 광대역**(프리앰블 76.6 MHz) → 셋 다 거리분해능 ~3.9 m 로 좋다. 데이터 점유(W1→W3)는 분해능을 거의 안 바꾼다(프리앰블이 고정 기준이라).
- **5G 는 상시(G1=SSB)면 7.2 MHz 로 병적으로 좁다**($\Delta R_b$ 41.6 m, 모노 등가 20.8 m) — 진짜 '5G 이중고'. 하지만 **G2·G3(NR-PRS)로 가면 98 MHz** 전대역 → $\Delta R_b$ 3.1 m 로 급반전.
- **LTE 는 L1=CRS(상시) → L2·L3=PRS(측위세션)** 로 기준신호가 바뀐다.

> **실증(X410)은 어디에?** X410 은 후보 파형을 **직접 송신(제어)** 하므로 전대역 상시 = **풀 모드(G3/L3/W3)** 에 해당한다. 상시(G1/L1/W1)는 '비협조 조명원일 때의 하한선'으로 읽으면 된다.

# §4. 수신기를 늘리면 — 코히어런트 배열 이득 (이상적 상한)

감시 배열 N 소자를 표적 방향으로 위상 맞춰 더하면 표적 √N 배·잡음 그대로 → **출력 SNR N배(+10log10 N dB)**. N=4 면 +6.0 dB. (직관: 여러 사람이 같은 소리를 방향 맞춰 함께 들으면 소리는 커지고 잡음은 그대로여서 더 또렷해진다.) 이는 **교과서적 배열이득이며, 여기서는 이상적 상한**이다: **완벽 조향**(표적 방위를 정확히 앎)·**소자 간 등분산 독립잡음**·보정오차 0 을 가정한다. 몬테카를로는 이 정합 빔포머와 동치인 형태로 √N 을 신호에 주입하고, 잡음쪽 전력보존(σ²)만 독립 실측한다. 실측 X410 은 조향 불일치·상호결합·동기오차로 **이 상한 이하**다.

![Rx 증설 RD 맵(5G 풀)](outputs/renders/anim/rd_rxbuildup_nr.gif)

<sub>거리-도플러 지도. 수신기 1→4 로 표적(흰 네모)이 잡음 위로 떠오른다. 0-도플러 세로 능선은 직접파 잔차(가드로 제외).</sub>

빔패턴도 N 으로 날카로워진다:

![배열 빔패턴](outputs/renders/anim/beampattern.gif)

<sub>감시 ULA 빔패턴이 N=1→4 로 좁아진다(표적 방위로 조향, 점선).</sub>

**검출확률 곡선(풀 모드).** 수신기를 늘릴수록 곡선이 왼쪽으로(더 약한 표적도 탐지) 이동:

![Pd vs SNR, N=1..4, 풀 모드](outputs/figures/report12_pd_curves.png)

![Pd 곡선 그려짐](outputs/renders/anim/pd_build_nr.gif)

<sub>N=1→4 순으로 그려가는 애니(5G 풀).</sub>

**필요 SNR(SNR50)과 수신기 증설 이득 — 풀 모드:**

| 모드 | N=1 | N=2 | N=3 | N=4 | N=4 이득 | 이상적 |
|---|---|---|---|---|---|---|
| W3 (WiFi) | 11.6 | 8.5 | 6.7 | 5.3 | **−6.2 dB** | −6.0 dB |
| L3 (LTE) | 13.7 | 10.7 | 8.9 | 7.6 | **−6.1 dB** | −6.0 dB |
| G3 (5G) | 11.2 | 8.2 | 6.3 | 5.4 | **−5.8 dB** | −6.0 dB |

이론(−6.0 dB)에 거의 정확히 붙는다 — **수신기 증설 이득은 조명원 종류와 무관한 배열 물리**다.

![감도 이득 vs 수신기 수](outputs/figures/report12_snr50_vs_n.png)

# §5. 몬테카를로 · Pfa — 얼마나 믿나 (검증)

Pd 는 확률이라 잡음을 **2,000번** 새로 뽑아(각 SNR·N·모드마다) 추정한다. 이 Pd 곡선은 **§6 에서 pyAPRiL 로 정확성을 검증한 그 표준 체인(ECA→거리도플러→CFAR)** 을, GPU 구현(`detection_gpu.py`)으로 대량 반복해 얻은 것이다 — pyAPRiL 자체가 Pd 곡선을 낸 것이 아니다. GPU 에 수십 트라이얼을 **배치로 한꺼번에** 올려(torch) 빠르게 반복한다(배치를 키우면 GPU 메모리를 많이 쓴다).

![몬테카를로 수렴](outputs/renders/anim/mc_converge_nr.gif)

<sub>시행수↑ 로 Pd 추정·95% 신뢰구간이 좁아진다.</sub>

**오경보율 — 정직하게(적대적 검증 반영).** report10 의 **파형(std)별** Pfa 교정을 적용했다. 하지만 이 교정은 표준별로만 되어 있어(점유/기준신호별 아님), 특히 **5G 상시(G1=SSB 7.2 MHz)** 처럼 대역이 아주 좁은 모드는 완전히 교정되지 않는다. 표적 없는 트라이얼로 잰 **경험적 Pfa 는 모드마다 다르고 목표 10⁻⁴ 와 정확히 같지 않다**(목표 대비 0.08~0.81배 — 최저는 G1). 따라서 **§3 의 모드 간 SNR50 차이 중 ~1 dB 안쪽의 미세한 차이는 이 불균일을 감안해 읽어야 한다** — 큰 차이(WiFi 광대역 vs 5G SSB, 수~십수 dB)는 대역폭 물리가 지배하므로 견고하다. 완전한 공정 비교는 각 모드의 **경험적 Pfa 축에 ROC 를 그려** 맞춰야 하며, 이는 향후 과제다.

![경험적 Pfa (9모드) — 목표 10⁻⁴ 대비 (완전 균일 아님)](outputs/figures/report12_pfa.png)

# §6. 선행 연구 속 위치 — 그리고 검증된 라이브러리 결합

패시브 레이더 드론탐지 논문 21편(WiFi·LTE·5G)을 정독해 보면, 이 프로젝트가 채우는 빈틈이 분명하다 (정직하게: 아래는 '아무도 정확히 이렇게 하지 않았다'는 뜻이지, 각 조각이 세계 최초라는 뜻은 아니다):

| 문헌의 빈틈 | 문헌 현황 | 이 프로젝트 |
|---|---|---|
| **조명원 간 공정 비교** | 조사한 논문은 **모두 단일 조명원**(5G만/LTE만/WiFi만). 표준 간 head-to-head 없음 | 9모드 W/L/G 공통 프로토콜 |
| **표적 RCS 를 실제로 모델** | 한 편도 드론 RCS 를 계산 안 함 — 외생 스칼라/Swerling 가정(−10~−13 dBsm) | SBR+PO, NACA-4 익형 프롭, 재질별 \|Γ\| |
| **점유율→Pd (고정 Pfa)** | LaSen 이 최선이나 monostatic·RMSE 채점 | G1/G2/G3 = SSB/PRS/CRS, '5G 이중고' 정량화 |
| **다중 Rx 이득 정량화** | **전부 최대 2채널**; N-Rx 코히어런트 이득을 잰 논문 0 | 1→4, 측정 5.8~6.4 dB vs 10log10 N |
| **통제·재현** | 전부 1회성 야외, 신뢰구간·공개데이터 없음 | 반무향 챔버 + K=2,000 MC + 재생성 파이프라인 |
| **정직한 결함 기록** | 명목 Pfa 를 그대로 신뢰 | 경험적 Pfa 불균일을 **결함으로 기록**(§5) |

<sub>⚠ **과장하지 않기**: 'SSB 가 상시신호'라는 관찰은 이미 문헌에 있다(예: Jopanya & Osorio, SPAWC 2025 · arXiv:2504.02641 — 5G NR SSB 패시브 바이스태틱 드론) — 우리 기여가 아니다. 우리 RCS 절대값(방위평균 **−18.4 dBsm**@3.5 GHz — 위 §2 의 이 자세값 -28.0 dBsm 은 **널이라 포락선 하단**이니 구분)도 소형드론 실측 포락선(−28~−16 dBsm)과 **정합하되 밝은 상단**(few-λ 공진영역에서 PO 가 절대레벨을 낙관적으로 잡을 소지, → report08)인 것이지 그 자체가 새 측정은 아니다. **다행히 헤드라인(5G 이중고·모드 비교)은 상대 결론이라 σ 가 밴드 내에서 움직여도 불변**이다. 이 프로젝트의 성격은 **통제된 재현가능 벤치마크 + 정직한 결함 노출**이다.</sub>

**뼈대는 주류 ISAC 아키텍처다.** 위 층별 구조(σ 외부 주입 + 표적/배경 채널 분리 + 표준 검출 체인)가 **선행의 주류**임을 확인했다(근거·출처: `prior_work/` pw01~03, `OPENSOURCE.md`):

| 구성 요소 | 같은 구조의 주류 선행 | 판정 |
|---|---|---|
| 표적/배경 채널 분리 `h = h_bg + h_target` | **NIST 5GNRad**·**3GPP Rel-19 ISAC**(오픈 MATLAB Putirf, arXiv:2606.07328)·**LAMBDA**(arXiv:2607.03826) | 동일 |
| 표적 밝기를 외부 σ 로 주입 | NIST 5GNRad·MATLAB·LAMBDA(CADFEKO) | 동일 |
| 환경 전파를 Sionna RT 로 담보 | Deterministic-Modeling(EuCAP 2026, arXiv:2603.28736)·SimART | 동일 |
| 검출 체인 ECA/CAF/CFAR | Wypich & Zielinski(Sensors 2026, USRP)·Demissie(IET RSN 2025) | 동일 |
| σ 를 확산계수 가정 대신 **드론 메쉬에서 SBR+PO 로 산출** | 대개 확산 S(Great-X, arXiv:2507.08716) 또는 점산란체 | **차이(강화)** |

즉 뼈대는 **NIST 5GNRad·3GPP·LAMBDA 와 같은 주류 ISAC 아키텍처**다. 덜 점유된 틈새는 세 결합 — ①상용 신호(WiFi/LTE/5G) **패시브 바이스태틱**(선행은 대개 능동/모노 또는 CSI 기반), ②드론 RCS 를 **SBR+PO 로 산출**(확산 S 가정 아님), ③**상시 vs 세션 9모드** 벤치마크.

> 🔧 **선행 방식·오픈소스로 검증(계층별, 전체 지도 `OPENSOURCE.md`·`prior_work/pw02`):**
> - **검출 체인 → pyAPRiL**(GPLv3, DVB-T/FM 실측 검증된 패시브 레이더 라이브러리). ECA/CAF/CFAR 는 **파형 무관**(reference I/Q 만 사용)이라 WiFi/LTE/5G 에 그대로 적합하다. 실제로 pyAPRiL 을 돌려 **NR/WiFi/LTE 3모드 모두 표적을 정답 거리빈에 검출**함을 확인했다(`benchmark/verify_pyapril.py`). ⚠ 단 §3~§5 의 **대량 몬테카를로 Pd 곡선은 pyAPRiL 이 아니라 GPU 구현(`detection_gpu.py`)**이 낸 것이다 — pyAPRiL 은 그 **표준 체인의 정확성을 검증**하고, Pd 곡선은 검증된 그 체인을 K=2,000회 GPU 로 대량 반복한 결과다.
> - **RCS → 자작 SBR+PO**(선행 BVH SBR+PO arXiv:2604.09243 과 같은 방법; 상용 CADFEKO·비공개 RadarSimPy 불채택), 절대값은 **실측 문헌 드론 RCS 로 앵커**(report08). **추적 → Stone Soup**, **실측 → OpenISAC+GNU Radio+X410**.
> - **파형·지연채널 → Sionna PHY**(report05, NMSE −135 dB). **아키텍처**(h=h_bg+h_target)는 **NIST 5GNRad·3GPP Rel-19**(오픈 구현 Putirf)와 동일.

> **실증(USRP X410)으로의 연결**: X410(TX4·RX4·400 MHz·12-bit)은 후보 파형을 **직접 송신**하므로 위 9모드를 통제 재현할 수 있고, 다중 RX 채널로 이 리포트의 Rx 이득을 실측 검증하는 자연스러운 테스트베드다. 단 시뮬(통제 챔버)↔실측(외부)은 환경이 1:1 이 아니라 **구조**(바이스태틱·기준+감시·같은 파형·디텍션 체인)만 같다.

# §7. 결론 — 탐지는 된다, 추적은 다음 일

1. **탐지가 된다.** 9모드 모두 충분한 SNR 에서 드론을 잡는다(Pd→1.0).
2. **조명원이 중요하다.** 상시라면 WiFi(광대역)가 유리, 5G(SSB)는 불리 — 단 PRS·제어면 5G 도 전대역. 실증(X410 제어)은 풀 모드에 해당.
3. **수신기를 늘리면 감도가 오른다.** Rx 1→4 로 필요한 SNR 이 5G 풀 기준 **5.8 dB** 낮아진다(이론 6 dB 와 일치).
4. **정직성.** 에코 지연은 Sionna `cir_to_time_channel`(상관 1.000 는 지연만 검증), 배열이득은 이상적 상한(√N 주입·잡음보존만 실측 1.000), Pfa 는 std별 명목 교정(모드 간 완전 균일은 아님). 모두 리포트에 명시했다.

> ### ▶ 다음 일 (future work): 추적
> 이 리포트는 **탐지**까지다. 위치·궤적을 잇는 **추적**은 감시 배열의 **각도(AoA)** 로 3D 관측가능성을 확보해야 한다(report11: 단일 수신기로는 3D 위치 불가). 본 실험이 쓴 **다중 수신기 배열이 그 출발점**이다.